# Retrieval-Augmented Generation (RAG)

**Domain:** LLM Inference, Training & Optimization  ·  **runnable:** yes

A refresher on grounding an LLM's answers in retrieved documents instead of its frozen weights.

## 1. What & Why

**What.** RAG bolts a *retriever* onto a *generator*. At query time you fetch the most relevant
chunks from an external corpus (your docs, a wiki, a database) and stuff them into the prompt as
context. The LLM then answers **grounded in that text** rather than only its pretrained memory.

**The problem it solves.** A base LLM has three chronic weaknesses:

- **Stale knowledge** — its weights freeze at a training cutoff; it can't know yesterday's release notes.
- **No private data** — it never saw your internal handbook, customer tickets, or codebase.
- **Hallucination** — asked something it doesn't know, it confidently makes things up.

RAG patches all three *without retraining*: you change the corpus, not the weights. It also gives you
**citations** (you know which chunk produced the answer) and **access control** (retrieve only docs the
user is allowed to see).

**When to reach for it.** Question-answering over a knowledge base, chat-with-your-docs, support bots,
internal search, anything where answers must trace back to a source of truth that changes faster than you
can fine-tune. **When not to:** if the knowledge is small and static, just put it in the system prompt; if
the task is a *skill* (tone, format, reasoning style) rather than *facts*, fine-tune instead — RAG injects
knowledge, not behavior.

## 2. Mental Model

Think of an **open-book exam**. The LLM is a smart student who studied months ago and has forgotten the
specifics. The retriever is the student flipping to the **three most relevant pages** of the textbook and
laying them open before writing the answer. The student's *reasoning* comes from training; the *facts*
come from the open pages on the desk.

```
                  ┌─────────────┐   embed    ┌──────────────┐
   Documents ────▶│  chunk +    │ ─────────▶ │  vector index │   (built once, offline)
                  │  embed      │            │  (the corpus) │
                  └─────────────┘            └──────┬───────┘
                                                    │ top-k nearest
   User query ──▶ embed ──▶ similarity search ──────┘
        │                                            │
        └────────────────────┐          ┌────────────┘ retrieved chunks
                             ▼          ▼
                       ┌─────────────────────┐
                       │  prompt =           │
                       │  context + question │ ──▶  LLM  ──▶  grounded answer (+ citations)
                       └─────────────────────┘
```

Two phases: an **offline indexing** phase (chunk → embed → store) and an **online query** phase
(embed query → retrieve → augment → generate). Retrieval quality is the ceiling on answer quality:
*garbage retrieved, garbage generated.*

## 3. Key Concepts

- **Chunking** — splitting documents into retrievable units (sentences, paragraphs, ~200–500 token
  windows, often with overlap). Too big = noisy context and wasted tokens; too small = lost context.
- **Embedding** — a model maps each chunk to a dense vector so that semantically similar text lands
  nearby. The *same* model must embed both corpus and query. (See the `vector-embeddings` notebook.)
- **Vector store / index** — holds the chunk vectors and answers nearest-neighbor queries, exactly or
  approximately (ANN: FAISS, HNSW, IVF). (See `faiss`, `chromadb`, the vector-DB notebooks.)
- **Similarity metric** — usually **cosine similarity** (angle between vectors) or dot product on
  normalized vectors. Top-`k` highest-scoring chunks are retrieved.
- **Augmentation** — assembling the prompt: a system instruction ("answer only from the context"),
  the retrieved chunks, and the user question.
- **Generation** — the LLM produces the answer conditioned on that prompt; ideally it **cites** the
  chunks and says "I don't know" when the context lacks the answer.
- **Top-`k`** — how many chunks to retrieve. More recall, more tokens, more distraction.
- **Reranking** — a second, more precise model re-scores the top-`k` candidates (see `rerankers`).
- **Hybrid search** — combine dense (embedding) retrieval with sparse keyword/BM25 retrieval; each
  catches what the other misses (exact IDs/codes vs. paraphrases).

## 4. Setup

The two worked examples below run on CPU with only **NumPy** — we build a tiny TF-IDF retriever from
scratch so the whole pipeline is transparent and dependency-free. The *generation* step is gated behind
an `ANTHROPIC_API_KEY` check: with a key it calls a real LLM; without one it falls back to an extractive
answer, so the notebook executes either way.

In production you'd swap the hand-rolled retriever for a real embedding model
(`sentence-transformers`, OpenAI/Cohere/Voyage embeddings) and a vector store (FAISS, Chroma, pgvector).

In [1]:
%pip install -q numpy

import os
import re
import numpy as np

print("numpy:", np.__version__)
print("ANTHROPIC_API_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))

Note: you may need to restart the kernel to use updated packages.
numpy: 2.5.0
ANTHROPIC_API_KEY set: False


## 5. Worked Examples

### Example 1 — A complete RAG retriever in NumPy

We implement the offline+online pipeline end to end: a small corpus, a TF-IDF embedding, cosine
similarity search, and prompt assembly. TF-IDF is a *sparse* embedding, but the retrieval mechanics
(vectorize → normalize → cosine → top-k) are identical to a dense pipeline — only the embedding model
changes.

In [2]:
# --- A tiny knowledge base (each string is one chunk) ---
CORPUS = [
    "RAG combines a retriever with a generator so the LLM answers from fetched documents.",
    "Chunking splits documents into small passages, often 200 to 500 tokens with some overlap.",
    "An embedding model maps text to a dense vector so similar meanings land near each other.",
    "FAISS is a library for fast approximate nearest-neighbor search over large vector sets.",
    "Cosine similarity scores the angle between two vectors and ignores their magnitude.",
    "Fine-tuning changes model weights to teach behavior or style, not fresh facts.",
    "A vector database stores embeddings and serves top-k nearest-neighbor queries at query time.",
    "Hybrid search blends dense embedding retrieval with sparse BM25 keyword matching.",
]

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

# --- Offline: build the TF-IDF "index" over the corpus ---
vocab = sorted({tok for doc in CORPUS for tok in tokenize(doc)})
vocab_idx = {w: i for i, w in enumerate(vocab)}
N = len(CORPUS)

# document frequency -> idf
df = np.zeros(len(vocab))
for doc in CORPUS:
    for w in set(tokenize(doc)):
        df[vocab_idx[w]] += 1
idf = np.log((N + 1) / (df + 1)) + 1.0  # smoothed idf

def embed(text):
    """Map text to a length-normalized TF-IDF vector."""
    vec = np.zeros(len(vocab))
    toks = tokenize(text)
    for w in toks:
        if w in vocab_idx:
            vec[vocab_idx[w]] += 1.0          # term frequency
    if toks:
        vec /= len(toks)                       # normalize tf by length
    vec *= idf                                 # weight by idf
    norm = np.linalg.norm(vec)
    return vec / norm if norm else vec

# Embed every chunk once -> the index matrix (n_chunks x vocab)
index = np.vstack([embed(doc) for doc in CORPUS])
print(f"Indexed {N} chunks over a {len(vocab)}-word vocabulary; index shape {index.shape}")

Indexed 8 chunks over a 87-word vocabulary; index shape (8, 87)


In [3]:
# --- Online: retrieve the top-k chunks for a query ---
def retrieve(query, k=3):
    q = embed(query)
    scores = index @ q              # cosine sim (all vectors are unit-normalized)
    top = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i]), CORPUS[i]) for i in top]

query = "How do I store embeddings and find the closest ones quickly?"
hits = retrieve(query, k=3)

print("Query:", query, "\n")
for rank, (i, score, text) in enumerate(hits, 1):
    print(f"{rank}. (chunk {i}, score {score:.3f}) {text}")

Query: How do I store embeddings and find the closest ones quickly? 

1. (chunk 6, score 0.304) A vector database stores embeddings and serves top-k nearest-neighbor queries at query time.
2. (chunk 4, score 0.268) Cosine similarity scores the angle between two vectors and ignores their magnitude.
3. (chunk 0, score 0.129) RAG combines a retriever with a generator so the LLM answers from fetched documents.


The top hits are the vector-database and FAISS chunks — exactly the relevant passages, retrieved by
meaning-overlap with no LLM involved yet. Notice the irrelevant fine-tuning chunk scores low. This is the
whole game: **retrieval decides what the model gets to see.**

### Example 2 — Augment the prompt and generate the answer

Now we assemble the retrieved chunks into a grounded prompt and generate. The LLM call is gated behind
`ANTHROPIC_API_KEY`; without a key we show the prompt and a simple extractive fallback, so the cell runs
anywhere.

In [4]:
def build_prompt(query, hits):
    context = "\n".join(f"[{i}] {text}" for i, _score, text in hits)
    return (
        "Answer the question using ONLY the context below. "
        "Cite the chunk numbers you used in [brackets]. "
        "If the context does not contain the answer, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )

prompt = build_prompt(query, hits)
print(prompt)

Answer the question using ONLY the context below. Cite the chunk numbers you used in [brackets]. If the context does not contain the answer, say you don't know.

Context:
[6] A vector database stores embeddings and serves top-k nearest-neighbor queries at query time.
[4] Cosine similarity scores the angle between two vectors and ignores their magnitude.
[0] RAG combines a retriever with a generator so the LLM answers from fetched documents.

Question: How do I store embeddings and find the closest ones quickly?
Answer:


In [5]:
# Generation step: real LLM if a key is present, extractive fallback otherwise.
if os.getenv("ANTHROPIC_API_KEY"):
    from anthropic import Anthropic  # pip install anthropic

    client = Anthropic()
    resp = client.messages.create(
        model="claude-haiku-4-5",     # small/cheap model is plenty for grounded QA
        max_tokens=256,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = resp.content[0].text
    print("LLM answer:\n", answer)
else:
    # Fallback: "answer" = highest-scoring retrieved chunk, with its citation.
    best_i, best_score, best_text = hits[0]
    print("[no ANTHROPIC_API_KEY — extractive fallback]\n")
    print(f"Answer (from chunk [{best_i}]): {best_text}")

[no ANTHROPIC_API_KEY — extractive fallback]

Answer (from chunk [6]): A vector database stores embeddings and serves top-k nearest-neighbor queries at query time.


Swap the retriever's TF-IDF `embed()` for a real embedding model and the corpus for thousands of chunks
in a vector store, and this is a production RAG system. Everything else — top-k, prompt assembly,
"answer only from context," citations — stays the same.

## 6. Gotchas & Pitfalls

- **Retrieval is the ceiling.** If the right chunk isn't in the top-k, the LLM cannot recover. Most "RAG
  is bad" complaints are really retrieval failures — measure recall@k before blaming the model.
- **Chunking dominates quality.** Splitting mid-sentence or mid-table destroys meaning; chunks too large
  bury the answer in noise and burn tokens. Tune chunk size/overlap to your content; respect structure
  (headings, paragraphs, code blocks).
- **Embedding mismatch.** You must embed the query and the corpus with the *same* model. Re-indexing with a
  new embedding model means re-embedding everything.
- **Lost in the middle.** LLMs attend most to the start and end of long contexts; a relevant chunk buried
  in the middle of many can be ignored. Retrieve fewer, better chunks (rerank) rather than dumping top-50.
- **Stale index.** The vector store is a snapshot. If source docs change, you must re-index, or you'll
  confidently serve outdated answers.
- **No grounding guardrail.** Without an explicit "answer only from the context / say you don't know"
  instruction, the model happily blends retrieved text with its own (possibly wrong) memory.
- **Semantic search misses exact tokens.** Dense retrieval can fail on error codes, SKUs, names, and
  acronyms — cases where you need an *exact* match. Add keyword/BM25 hybrid search for those.
- **Cost and latency.** Every query now pays for an embedding call, a vector search, and a bigger prompt.
  Long contexts are slower and pricier — retrieve tightly.

## 7. When to Use vs Alternatives

| Approach | Best for | Cost / effort | Limitation |
|---|---|---|---|
| **RAG** | Facts that change or are private; need citations & access control | Low: just re-index docs | Quality capped by retrieval; added latency/tokens |
| **Long-context prompt** (stuff all docs in) | Small, static knowledge that fits the window | Trivial | Doesn't scale; expensive per call; "lost in the middle" |
| **Fine-tuning** | Teaching *behavior*: tone, format, reasoning style, narrow task | High: data + training + redeploy on every change | Bakes in facts that go stale; weak at precise recall; no citations |
| **Tool/function calling** | Live, structured data (DB rows, APIs, calculations) | Medium | Needs well-defined tools; not for fuzzy doc search |
| **Agentic / iterative RAG** | Multi-hop questions needing several retrieval rounds | High: orchestration, more LLM calls | Slower, harder to debug, more failure modes |

**Rules of thumb:** Need *knowledge*? RAG. Need *behavior*? Fine-tune. Knowledge tiny and static? Just
prompt it. Need live computed/structured data? Tools. These compose — fine-tune a model for your domain's
tone **and** use RAG to feed it current facts. Start with the simplest (prompt → RAG → rerank/hybrid →
agentic) and add machinery only when evaluation shows you need it.

## 8. Resources

- **Original RAG paper** — Lewis et al., 2020, "Retrieval-Augmented Generation for Knowledge-Intensive NLP
  Tasks": <https://arxiv.org/abs/2005.11401>
- **"Lost in the Middle"** — Liu et al., on how context position affects LLM recall:
  <https://arxiv.org/abs/2307.03172>
- **LangChain RAG tutorial** — end-to-end pipeline with loaders, splitters, and vector stores:
  <https://python.langchain.com/docs/tutorials/rag/>
- **LlamaIndex docs** — data framework purpose-built for RAG over your own data:
  <https://docs.llamaindex.ai/>
- **Pinecone "Retrieval Augmented Generation" guide** — practical, well-illustrated overview:
  <https://www.pinecone.io/learn/retrieval-augmented-generation/>

**Related notebooks in this domain:** `vector-embeddings`, `semantic-search`, `faiss`, `chromadb`,
`rerankers`, `graphrag`, `llamaindex`.